# PRÁCTICA PR4 (PARTE GUIADA) | ANOVA PRyES
## Trabajo con temperaturas mínimas y máximas de España (fuente: AEMET)
Objetivos de aprendizaje
- Realizar un ANOVA con un factor
- Realizar un ANOVA con dos factores sin interacción
- Realizar un ANOVA con dos factores con interacción

### 0: Carga de librerías necesarias

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from more_itertools import locate

from anova import anova1, anova2

In [ ]:
# asegura que pandas mostrará todo el dataframe completo (100 filas)
# necesario para visualizar bien modelo de medias de ANOVA
pd.set_option('display.max_rows', 100)

### 1: Carga de datos

In [ ]:
# leemos los datos del fichero con las temperaturas usando pandas
# Los datos están en ºC

df_orig = pd.read_csv("tableTemperaturesAnova.csv")

# parseamos la fecha (cadena) para que sea un datetime con formato dd/mm/yyyy
df_orig["DATE"] = pd.to_datetime(df_orig["DATE"], format='%d/%m/%Y') 
print('Tamaño de df_orig con los datos cargados:', df_orig.shape)
print('Dataframe con todos los datos leidos (filas: días, columnas: variables temperaturas):')
df_orig

### 2. Análisis exploratorio básico

In [ ]:
# Diagramos de cajas de la TMIN de cada ciudad disponible
df_orig.boxplot(
    column = "TMIN", 
    by = "TOWN",
    figsize=(20,5), 
    rot=45, 
)

In [ ]:
## Diagramas de cajas de TMIN con el mes y la estación del año
fig, axes = plt.subplots(
    nrows=1, 
    ncols=2, 
    figsize=(20,5)
)
df_orig.boxplot(
    column = "TMIN", 
    by = "MONTH",
    figsize=(20,5), 
    rot=45, 
    ax = axes[0]
)
df_orig.boxplot(
    column = "TMIN", 
    by = "SEASON",
    figsize=(20,5), 
    rot=45, 
    ax = axes[1]
)

axes[0].set_title('TMIN en función de MONTH')
axes[1].set_title('TMIN en función de SEASON')
fig.suptitle('Diagramas de cajas TMIN')

## 3. ANOVA de un factor

### 3.1 Ejecuta ANOVA 'TMIN ~ C(YEAR)'

In [ ]:
# Selección de los datos
df = (
    df_orig
    .loc[lambda a:
        (a.YEAR >= 1997) & (a.YEAR <= 2010) &
        a.TOWN.isin([3946, 237, 421]) &
        (a.MONTH >=2 ) & (a.MONTH <= 10)
    ]
)

# Creación de la fórmula

salida = 'TMIN'
factor = 'YEAR'
formula = f'{salida} ~ C({factor})'
print('Fórmula: ', formula)

# Cálculo ANOVA

tabla_anova, modelo_medias = anova1(formula, df)

# Mostrar tabla Anova

print('Tabla ANOVA:')
print(tabla_anova)
print('\n')
print('Grados de libertad totales: ' , tabla_anova.df.sum())

# Mostrar modelo de medias

print('Modelo de medias:')
print(modelo_medias)

### 3.2 Ejecuta ANOVA 'TMIN ~ C(MONTH)'

In [ ]:
# Selección de los datos
df = (
    df_orig
    .loc[lambda a:
        (a.YEAR >= 1997) & (a.YEAR <= 2010) &
        a.TOWN.isin([3946, 237, 421]) &
        (a.MONTH >=2 ) & (a.MONTH <= 10)
    ]
)

# Creación de la fórmula

salida = 'TMIN'
factor = 'MONTH'
formula = f'{salida} ~ C({factor})'
print('Fórmula: ', formula)

# Cálculo ANOVA

tabla_anova, modelo_medias = anova1(formula, df)

# Mostrar tabla anova

print('Tabla ANOVA:')
print(tabla_anova)
print('\n')
print('Grados de libertad totales: ' , tabla_anova.df.sum())

# Mostrar modelo de medias

print('Modelo de medias:')
print(modelo_medias)

## 4. ANOVA de dos factores

### 4.1 Ejecuta ANOVA 'TMAX ~ C(MONTH) + C(TOWN)'  (+: sin interacción)

In [ ]:
# Selección de los datos

df  = (
    df_orig
    .loc[lambda a:
        (a.YEAR >= 1997) & (a.YEAR <= 2010) &
        a.TOWN.isin([3946, 237, 421]) &
        (a.MONTH >=1 ) & (a.MONTH <= 12) # Todos los meses
    ]
)

# Creación de la fórmula (sin interacción)

salida = 'TMAX'
factor_1 = 'MONTH'
factor_2 = 'TOWN'
formula = f'{salida} ~ C({factor_1}) + C({factor_2})'
print('Fórmula: ', formula)

# Cálculo ANOVA 2

tabla_anova, modelo_medias = anova2(formula, df)

# Mostrar tabla anova

print('Tabla ANOVA:')
print(tabla_anova)
print('\n')
print('Grados de libertad totales: ' , tabla_anova.df.sum())

# Mostrar modelo de medias

print('Modelo de medias:')
print(modelo_medias)

In [ ]:
# Mostrar modelo de medias y sus coeficientes alfa y beta

print('Modelo de medias:')
print(modelo_medias)

ind_factor_1 = np.where(modelo_medias.index.str.startswith(factor_1))[0].tolist()
ind_factor_2 = np.where(modelo_medias.index.str.startswith(factor_2))[0].tolist()

gr_factor_1 = modelo_medias.index[ind_factor_1].str.replace('.*_', '', regex =True)
gr_factor_2 = modelo_medias.index[ind_factor_2].str.replace('.*_', '', regex =True)


fig = plt.figure(1, figsize=(20, 3))
plt.subplot(1,3,1)
plt.bar(modelo_medias.index[0], modelo_medias.Mean.iloc[0])
plt.ylabel('Media global')

plt.subplot(1,3,2)
plt.bar(gr_factor_1, modelo_medias.Mean.iloc[ind_factor_1])
plt.ylabel('Coeficientes alfa')
plt.xlabel(factor_1)

plt.subplot(1,3,3)
plt.bar(gr_factor_2, modelo_medias.Mean.iloc[ind_factor_2])
plt.ylabel('Coeficientes beta')
plt.xlabel(factor_2)
plt.show()

# gráfica 3D
X, Y = np.mgrid[0:len(ind_factor_1), 0:len(ind_factor_2)]

# evalua el modelo de medias en cada punto (x_i,y_j)
Z = np.zeros(X.shape)
for i in range(X.shape[0]):
    for j in range(X.shape[1]):
         Z[i,j] =  modelo_medias.Mean.iloc[0] + modelo_medias.Mean.iloc[ind_factor_1].iloc[i] +  modelo_medias.Mean.iloc[ind_factor_2].iloc[j]
 
Z_sin_inter = Z

fig = plt.figure(2, figsize=(8, 8))
  
ax1 = plt.axes(projection='3d')
ax1.set_xticks(range(len(ind_factor_1)))
ax1.set_yticks(range(len(ind_factor_2)))

ax1.set_xticklabels(gr_factor_1)
ax1.set_yticklabels(gr_factor_2)
ax1.plot_wireframe(X, Y, Z, color='red')
ax1.stem(X.flatten(), Y.flatten(), Z.flatten(), basefmt='none')
plt.xlabel(factor_1); plt.ylabel(factor_2)
plt.title(f'Modelo de Medias ANOVA:  {formula}')
plt.show()

### 4.2 Ejecuta ANOVA 'TMAX ~ C(MONTH) * C(TOWN)'  (*: con interacción) 

In [ ]:
# Selección de los datos

df  = (
    df_orig
    .loc[lambda a:
        (a.YEAR >= 1997) & (a.YEAR <= 2010) &
        a.TOWN.isin([3946, 237, 421]) &
        (a.MONTH >=1 ) & (a.MONTH <= 12) # Todos los meses
    ]
)

# Creación de la fórmula (con interacción)

salida = 'TMAX'
factor_1 = 'MONTH'
factor_2 = 'TOWN'
formula = f'{salida} ~ C({factor_1}) * C({factor_2})'
print('Fórmula: ', formula)

# Cálculo ANOVA 2

tabla_anova, modelo_medias = anova2(formula, df)

# Mostrar tabla anova

print('Tabla ANOVA:')
print(tabla_anova)
print('\n')
print('Grados de libertad totales: ' , tabla_anova.df.sum())

# Mostrar modelo de medias

print('Modelo de medias:')
print(modelo_medias)

In [ ]:
# Clacular coeficientes interacción

modelo_medias_not_inter = (
    modelo_medias
    .loc[lambda a: ~a.index.str.contains(":")]
)

modelo_medias_inter = (
    modelo_medias
    .loc[lambda a: a.index.str.contains(":")]
    .assign(
        Month = lambda a: a.index.str.split(" : ").str[0],
        Town = lambda a: a.index.str.split(" : ").str[1]
    )
)

# Mostrar modelo de medias y sus coeficientes alfa, beta y theta

print('Modelo de medias:')
print(modelo_medias)

ind_factor_1 = np.where(modelo_medias_not_inter.index.str.startswith(factor_1))[0].tolist()
ind_factor_2 = np.where(modelo_medias_not_inter.index.str.startswith(factor_2))[0].tolist()

gr_factor_1 = modelo_medias_not_inter.index[ind_factor_1].str.replace('.*_', '', regex =True)
gr_factor_2 = modelo_medias_not_inter.index[ind_factor_2].str.replace('.*_', '', regex =True)


fig = plt.figure(1, figsize=(20, 3))
plt.subplot(1,3,1)
plt.bar(modelo_medias_not_inter.index[0], modelo_medias_not_inter.Mean.iloc[0])
plt.ylabel('Media global')

plt.subplot(1,3,2)
plt.bar(gr_factor_1, modelo_medias_not_inter.Mean.iloc[ind_factor_1])
plt.ylabel('Coeficientes alfa')
plt.xlabel(factor_1)

plt.subplot(1,3,3)
plt.bar(gr_factor_2, modelo_medias_not_inter.Mean.iloc[ind_factor_2])
plt.ylabel('Coeficientes beta')
plt.xlabel(factor_2)
plt.show()

fig = plt.figure(1, figsize=(40, 3))
plt.subplot(1,3,1)
plt.bar(modelo_medias_inter.index, modelo_medias_inter.Mean)
plt.ylabel('Coeficientes theta')
plt.xticks(rotation=90)

# gráfica 3D
X, Y = np.mgrid[0:len(ind_factor_1), 0:len(ind_factor_2)]

# evalua el modelo de medias en cada punto (x_i,y_j)
Z = np.zeros(X.shape)
for i in range(X.shape[0]):
    for j in range(X.shape[1]):

        factor_1_name = modelo_medias_not_inter.iloc[ind_factor_1].index[i]
        factor_2_name = modelo_medias_not_inter.iloc[ind_factor_2].index[j]

        interaction = modelo_medias_inter.loc[lambda a: (a.Month == factor_1_name) & (a.Town == factor_2_name)].Mean.iloc[0]

        Z[i,j] =  modelo_medias_not_inter.Mean.iloc[0] + modelo_medias_not_inter.Mean.iloc[ind_factor_1].iloc[i] +  modelo_medias_not_inter.Mean.iloc[ind_factor_2].iloc[j]+ interaction
   
Z_con_inter = Z

fig = plt.figure(2, figsize=(8, 8))
  
ax1 = plt.axes(projection='3d')
ax1.set_xticks(range(len(ind_factor_1)))
ax1.set_yticks(range(len(ind_factor_2)))

ax1.set_xticklabels(gr_factor_1)
ax1.set_yticklabels(gr_factor_2)
ax1.plot_wireframe(X, Y, Z, color='green')
ax1.stem(X.flatten(), Y.flatten(), Z.flatten(), basefmt='none')
plt.xlabel(factor_1); plt.ylabel(factor_2);
plt.title(f'Modelo de Medias ANOVA:  {formula}')
plt.show()

In [ ]:
# comparación modelo medias con y sin interacción

fig = plt.figure(1, figsize=(8, 8))
  
ax1 = plt.axes(projection='3d')
ax1.set_xticks(range(len(ind_factor_1)))
ax1.set_yticks(range(len(ind_factor_2)))

ax1.set_xticklabels(gr_factor_1)
ax1.set_yticklabels(gr_factor_2)
ax1.plot_wireframe(X, Y, Z_sin_inter, color='red', label = 'Sin interacción')
ax1.plot_wireframe(X, Y, Z_con_inter, color='green', label = 'Con interacción')
ax1.view_init(azim=-60, elev=50)
plt.xlabel(factor_1); plt.ylabel(factor_2); plt.legend()
plt.title(f'Modelo de Medias ANOVA (con y sin interacción) ')
plt.show()

# diferencias entre ammbos modelos
fig = plt.figure(2, figsize=(8, 8))
ax1 = plt.axes(projection='3d')
ax1.set_xticks(range(len(ind_factor_1)))
ax1.set_yticks(range(len(ind_factor_2)))
ax1.set_xticklabels(gr_factor_1)
ax1.set_yticklabels(gr_factor_2)
ax1.plot_wireframe(X, Y, Z_con_inter - Z_sin_inter, color='blue')
ax1.stem(X.flatten(), Y.flatten(), (Z_con_inter - Z_sin_inter).flatten(), basefmt='none')
ax1.view_init(azim=-60, elev=50)
plt.xlabel(factor_1); plt.ylabel(factor_2);
plt.title(f'Modelo de Medias ANOVA (diferencias con y sin interacción) ')
plt.show()